# 1. DATA UNDERSTANDING

In [0]:
import pyspark.sql.functions as F

##  1.1 Carga Inicial de Datos

In [0]:
dfAccesoInternetOficiales = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Instituciones-Educativas-Oficiales-de-Boyacá-con-Acceso-a-Internet.csv",
    header=True, inferSchema=True
)
dfAccesoInternetOficiales.createOrReplaceTempView("dfAccesoInternetOficiales")
display(dfAccesoInternetOficiales)

## 1.2 Descripción de Datos

In [0]:
#Revisar las columnas para ver que carachas andamos viendo
dfAccesoInternetOficiales.printSchema()

In [0]:
#Cambiar el nombre de las columnas con - , en minúscula y sin tildes

df01 = dfAccesoInternetOficiales.withColumnRenamed("CÓDIGO DEPARTAMENTO","codigo-departamento").withColumnRenamed("DEPARTAMENTO","departamento").withColumnRenamed("PROVINCIA","provincia").withColumnRenamed("CÓDIGO MUNICIPIO","codigo-municipio").withColumnRenamed("MUNICIPIO","municipio").withColumnRenamed("CODIGO DANE INSTITUCION EDUCATIVA","codigo-dane-institucion-educativa").withColumnRenamed("NOMBRE INSTITUCION EDUCATIVA","institucion-educativa").withColumnRenamed("CODIGO DANE SEDE","codigo-dane-escuela").withColumnRenamed("NOMBRE SEDE EDUCATIVA","escuela").withColumnRenamed("ZONA","zona").withColumnRenamed("PROYECTOS DE CONECTIVIDAD 2024","proyectos-de-conectividad-2024").withColumnRenamed("OPERADOR","operador").withColumnRenamed("ESTADO","estado").withColumnRenamed("MEDIO DE ENLACE","medio-de-enlace").withColumnRenamed("ANCHO DE BANDA DE SUBIDA (MB)","ancho-de-banda-de-subida").withColumnRenamed("ANCHO DE BANDA DESCARGA (MB)","ancho-de-banda-descarga").withColumnRenamed("FINALIZACIÓN DEL CONTRATO","finalizacion-del-contrato").withColumnRenamed("LATITUD","latitud").withColumnRenamed("LONGITUD","longitud")


In [0]:
df01.display()

In [0]:
df01.select(F.col("departamento")).distinct().show(truncate=False)

df01.select(F.col("provincia")).distinct().show(truncate=False)

df01.select(F.col("municipio")).distinct().show(truncate=False)


Se decide eliminar la columna de departamento y código de departamento como solamente vamos a trabajar con Boyacá.

In [0]:
df01.drop("departamento","provincia","municipio").show()

In [0]:
df01.select(F.col("proyectos-de-conectividad-2024")).distinct().show(truncate=False)

df01.select(F.col("operador")).distinct().show(truncate=False)

df01.select(F.col("estado")).distinct().show(truncate=False)

df01.select(F.col("medio-de-enlace")).distinct().show(truncate=False)

df01.select(F.col("finalizacion-del-contrato")).distinct().show(truncate=False)

Se observa que para 2024 se tienen 4 proyectos vigentes y existen 3 operadores. En el estados tenemos 4 posibles valores, el medio de enlace puede llegar a ser un indicativo de calidad y de la zona (si es de díficil acceso) para la conectividad de la instición. Se debe revisar porqué existe NA y NULL a tiempo. Y por último la finalización de contrato que se asume que es NULL para los que siguen vigente o para instituciones que no tienen aún contrato. 

Adicionalmente se observa como en estado está "PENDIENTE INICIO OPERACIÓN" Y "PENDIENTE INICIO DE OPERACIÓN" que es lo mismo, por lo que se procede a limpiar

In [0]:
df01 = df01.withColumn("estado",F.when((F.col("estado") == "PENDIENTE INICIO OPERACIÓN") |(F.col("estado") == "PENDIENTE INICIO DE OPERACIÓN"),"PENDIENTE INICIO OPERACION").otherwise(F.col("estado")))

df01.select(F.col("estado")).distinct().show(truncate=False)

#Ya quedó limpio :D

In [0]:
df01.groupBy(F.col("proyectos-de-conectividad-2024"),F.col("operador")).count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
df01.groupBy(F.col("proyectos-de-conectividad-2024"),F.col("estado")).count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
df01.groupBy("estado").count().orderBy("count", ascending=False).show(truncate=False)

De los dos tres bloques anteriores de código se observa que los que están sin proyecto no tienen servicio ni operador. Y se observa como hay más proyectos pendientes de inicio de operación y directamente sin servicio que los operativos. Demostrando una gran brecha de acceso a internet en ambientes educativos para el año 2024.

In [0]:
df01.groupBy(F.col("medio-de-enlace"),F.col("proyectos-de-conectividad-2024")).count().orderBy("count", ascending=False).show(truncate=False)

Se observa como los medios de enlace se encuentran NULL y NA en los que no tienen proyecto. Se propone unificar bajo "SIN MEDIO" para este caso.

Mientras que para los "NA" de los que si tienen proyecto, se propone en dejarlos como "SIN DATOS" por lo que estós son los que están pendientes de implementación.

In [0]:
df01 = df01.withColumn("medio-de-enlace",
    F.when((F.col("proyectos-de-conectividad-2024") == "SIN PROYECTO") &(F.col("medio-de-enlace").isNull() | (F.col("medio-de-enlace") =="NA")),"SIN MEDIO"
    ).when((F.col("proyectos-de-conectividad-2024") != "SIN PROYECTO") &(F.col("medio-de-enlace") == "NA"),"SIN DATOS").otherwise(F.col("medio-de-enlace"))
)

In [0]:
df01.display()